# UC severity study – slow experiments on a Colab GPU

# UC 严重程度研究——在 Colab GPU 上运行耗时的实验

This notebook runs the two slow ViLT VQA trainings (`main` and `kvasir`) on a free Colab T4 GPU, plus optional extras. Before starting: upload `colab_bundle.zip` to `My Drive/UC_project/`, then choose **Runtime → Change runtime type → T4 GPU → Save**. Run the cells from top to bottom with Shift + Enter.

本笔记本在免费的 Colab T4 GPU 上运行两个耗时的 ViLT VQA 训练（`main` 和 `kvasir`），以及可选的附加实验。开始之前：把 `colab_bundle.zip` 上传到 `我的云端硬盘/UC_project/`，然后选择 **代码执行程序 → 更改运行时类型 → T4 GPU → 保存**。用 Shift + Enter 从上到下依次运行各单元格。

Results are copied to `My Drive/UC_project/colab_results.zip` after every run, so nothing is lost if Colab disconnects. At the end, download that file and unzip it into the project folder on the laptop.

每次运行结束后，结果都会复制到 `我的云端硬盘/UC_project/colab_results.zip`，因此即使 Colab 断开也不会丢失。最后下载该文件，并解压到笔记本电脑上的项目文件夹中。

## 1. Check the GPU

## 1. 检查 GPU

In [ ]:
!nvidia-smi  # Show the GPU Colab assigned; it should say Tesla T4 or better.

## 2. Connect Google Drive and unpack the bundle

## 2. 连接 Google Drive 并解压文件包

A Google sign-in window opens; approve it yourself. The project is unpacked to Colab's fast local disk (`/content/uc`).

会弹出 Google 登录窗口，请自己完成授权。项目会被解压到 Colab 的高速本地磁盘（`/content/uc`）。

In [ ]:
import os  # Folder and path tools.
import zipfile  # Read and write ZIP archives.
from google.colab import drive  # Colab's Google Drive helper.

drive.mount("/content/drive")  # Connect your Drive to this session.
DRIVE_DIR = "/content/drive/MyDrive/UC_project"  # Drive folder that holds the bundle and receives the results.
WORK_DIR = "/content/uc"  # Working copy on Colab's local disk.
with zipfile.ZipFile(f"{DRIVE_DIR}/colab_bundle.zip") as zf:  # Open the uploaded bundle.
    zf.extractall(WORK_DIR)  # Unpack scripts, data and archives.
os.chdir(WORK_DIR)  # Run everything from the project root, as on the laptop.
print(sorted(os.listdir(WORK_DIR)))  # Expect colab, data, requirements.txt and scripts.

## 3. Match the library versions

## 3. 统一库版本

The other models were trained with transformers 5.20.2 and scikit-learn 1.9.1. Colab's own CUDA build of torch is kept; do not reinstall torch, or the GPU support is lost.

其他模型是用 transformers 5.20.2 和 scikit-learn 1.9.1 训练的。保留 Colab 自带的 CUDA 版 torch；不要重装 torch，否则会失去 GPU 支持。

In [ ]:
!pip install -q "transformers==5.20.2" "scikit-learn==1.9.1"  # Install the versions used for the other models.

In [ ]:
import json  # Save the environment description.
import platform  # Python version.
import sklearn  # scikit-learn, for its version number.
import torch  # PyTorch, for its version and the GPU check.
import transformers  # Hugging Face transformers, for its version.

assert torch.cuda.is_available(), "No GPU: choose Runtime -> Change runtime type -> T4 GPU"  # Stop here if the session has no GPU.
env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,  # Library versions.
       "scikit-learn": sklearn.__version__, "gpu": torch.cuda.get_device_name(0), "where": "Google Colab"}  # Hardware and place.
os.makedirs("results/training_logs", exist_ok=True)  # Folders for results and logs.
with open("results/environment_colab.json", "w") as f:  # Record the environment for the report.
    json.dump(env, f, indent=2)  # Write it as readable JSON.
print(env)  # Show it; transformers must be 5.20.2 and scikit-learn 1.9.1.

## 4. Rebuild the images and confirm the split is unchanged

## 4. 重建图像并确认划分未变

`prepare_dataset.py` extracts the 851 HyperKvasir images and rewrites `manifest.csv`. Colab's JPEG decoder can give slightly different pixel values from the laptop's, so the rebuilt colour averages (and possibly hashes) may differ a little. What matters is that the image files are byte-for-byte the same; the committed split files are then restored from the bundle, so Colab trains on exactly the split every other model used.

`prepare_dataset.py` 会解压 851 张 HyperKvasir 图像并重写 `manifest.csv`。Colab 的 JPEG 解码器给出的像素值可能与笔记本电脑略有不同，因此重建的平均颜色（也可能包括哈希值）会有细微差异。关键是图像文件本身逐字节相同；随后从文件包中恢复已提交的划分文件，使 Colab 使用与其他所有模型完全相同的划分进行训练。

In [ ]:
import csv  # Read the two manifests.
import hashlib  # SHA-256 fingerprints.
import shutil  # Copy files.
import PIL  # Pillow, only to print its version.

def sha256(path):  # Fingerprint of one file.
    return hashlib.sha256(open(path, "rb").read()).hexdigest()  # Hash of the file's bytes.

def read(path):  # Load a manifest as {image path: row}.
    with open(path, encoding="utf-8") as f:  # Open the CSV file.
        return {r["path"]: r for r in csv.DictReader(f)}  # One entry per image.

shutil.copy("data/processed/manifest.csv", "/content/manifest_original.csv")  # Keep a copy of the committed manifest.
!python scripts/prepare_dataset.py > results/training_logs/logs_prepare_dataset_colab.txt 2>&1  # Extract and audit the images; rewrites the manifest.
!python scripts/prepare_kvasir_vqa.py > results/training_logs/logs_prepare_kvasir_vqa_colab.txt 2>&1  # Extract Kvasir-VQA and rebuild its question table.
old, new = read("/content/manifest_original.csv"), read("data/processed/manifest.csv")  # Committed manifest vs the one rebuilt on Colab.
assert old.keys() == new.keys(), "Different image lists - stop and tell Claude Code"  # Both must list the same 851 images.
print("Pillow", PIL.__version__, "| images:", len(old))  # Decoder version and image count.
for col in ["grade", "split", "group", "width", "height", "mean_r", "mean_g", "mean_b", "sha256", "dhash"]:  # Check every column.
    print(f"{col:7s} differs for {sum(old[p][col] != new[p][col] for p in old)} images")  # Number of images whose value changed.
assert all(old[p]["sha256"] == sha256(p) for p in old), "Image bytes differ - stop and tell Claude Code"  # The image files themselves must be identical.
with zipfile.ZipFile(f"{DRIVE_DIR}/colab_bundle.zip") as zf:  # Open the uploaded bundle again.
    for name in ["manifest.csv", "similar_pairs.csv", "kvasir_vqa_uc_qa.csv", "dataset_summary.json", "kvasir_vqa_summary.json"]:  # All committed data files.
        zf.extract(f"data/processed/{name}", ".")  # Overwrite the Colab-rebuilt copy with the committed one.
restored = sha256("data/processed/manifest.csv") == sha256("/content/manifest_original.csv")  # Is the committed split back in place?
print("Committed split restored:", restored)  # Must print True before training.
assert restored, "Restore failed - stop and tell Claude Code"  # Never train on a different split.

## 5. Helper: copy the results to Drive

## 5. 辅助函数：把结果复制到 Drive

In [ ]:
import shutil  # Copy files.

def save_to_drive():  # Zip the whole results folder and copy it to Drive.
    shutil.make_archive("/content/colab_results", "zip", WORK_DIR, "results")  # Build colab_results.zip with results/ inside.
    shutil.copy("/content/colab_results.zip", DRIVE_DIR)  # Overwrite the copy on Drive with the newest version.
    print("Saved to", DRIVE_DIR + "/colab_results.zip")  # Confirm where it went.

save_to_drive()  # First save (environment file and data logs).

## 6. ViLT VQA – mode `main` (grading question only)

## 6. ViLT VQA——`main` 模式（只问分级问题）

In [ ]:
!python scripts/train_vqa.py --mode main --device cuda 2>&1 | tee results/training_logs/logs_vqa_main_colab.txt  # Train and test ViLT on the GPU, keeping a log.
save_to_drive()  # Copy the new result to Drive right away.

## 7. ViLT VQA – mode `kvasir` (ordinal questions + Kvasir-VQA questions)

## 7. ViLT VQA——`kvasir` 模式（有序问题 + Kvasir-VQA 问题）

In [ ]:
!python scripts/train_vqa.py --mode kvasir --device cuda 2>&1 | tee results/training_logs/logs_vqa_kvasir_colab.txt  # Train with Kvasir-VQA questions added, on the GPU.
save_to_drive()  # Copy the new result to Drive right away.

## 8. Optional: 6-class task for the best CNN and the best ViT

## 8. 可选：对最佳 CNN 和最佳 ViT 做六分类任务

Set `RUN_SIX = True` only if you want this optional experiment (ResNet-50 and Swin-Tiny, same learning rates as the 3-class runs).

只有在需要这个可选实验时才把 `RUN_SIX` 设为 `True`（ResNet-50 和 Swin-Tiny，学习率与三分类运行相同）。

In [ ]:
RUN_SIX = False  # Change to True to run the optional 6-class experiment.
if RUN_SIX:  # Skip everything below unless requested.
    !python scripts/train_classifier.py --model microsoft/resnet-50 --name resnet50 --family CNN --lr 5e-4 --task six --device cuda 2>&1 | tee results/training_logs/logs_six_resnet50_colab.txt  # Best CNN, 6 classes.
    !python scripts/train_classifier.py --model microsoft/swin-tiny-patch4-window7-224 --name swin_tiny --family ViT --lr 5e-5 --task six --device cuda 2>&1 | tee results/training_logs/logs_six_swin_tiny_colab.txt  # Best ViT, 6 classes.
    save_to_drive()  # Copy the new results to Drive.

## 9. Finish

## 9. 完成

Download `My Drive/UC_project/colab_results.zip` and unzip it into the project folder on the laptop, so that `results/three/vilt_vqa_main/` and `results/three/vilt_vqa_kvasir/` sit next to the existing results. Then tell Claude Code that the Colab results are in place. You can now close the session with **Runtime → Disconnect and delete runtime**.

下载 `我的云端硬盘/UC_project/colab_results.zip`，并解压到笔记本电脑上的项目文件夹中，使 `results/three/vilt_vqa_main/` 和 `results/three/vilt_vqa_kvasir/` 与现有结果放在一起。然后告诉 Claude Code：Colab 结果已放好。现在可以通过 **代码执行程序 → 断开连接并删除运行时** 关闭会话。

In [ ]:
!ls -R results | head -40  # List what will be in colab_results.zip.